# MiniMed Prime Colab: Build Golden-Seed Data

This notebook builds MedReason seed data in Colab with the real retrieval stack, then splits the output into `goldish`, `silver`, and `reject` tiers.

What this notebook does:
- materializes the project source from Drive, Git, or a zip bundle
- installs the minimal dependency set for seed building
- resolves assets from Google Drive or optionally downloads missing PrimeKG, SapBERT, and MedCPT checkpoints
- runs `scripts/prepare_medreason_seed.py` for train and validation
- runs `scripts/filter_seed_quality.py` to split `goldish / silver / reject`
- optionally builds TRM arrays from the `goldish` split

Quality contract used here:
- `goldish`: non-empty `gold_edge_ids`, edge mapper backend in `direct` or `llm_*`, `entity_linker_backend == scispacy_umls`, and PrimeKG is not an explicit empty-graph fallback
- `silver`: usable seed rows, but at least one quality gate failed
- `reject`: missing gold edges, missing evidence edges, invalid rows, or explicit empty-graph retrieval

Important:
- This notebook produces `goldish` candidates, not literal human-reviewed gold.
- If you keep `EDGE_MAPPER=auto` or `EDGE_MAPPER=llm` with a remote model such as `gpt-4o-mini`, you must provide the matching API key in Colab Secrets.
- For the best quality, keep PrimeKG, SapBERT, MedCPT, and scispaCy all real.


In [ ]:
#@title 1. Config
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
PROJECT_SOURCE_MODE = "drive_folder"  #@param ["drive_folder", "git", "zip_bundle"]
PROJECT_DRIVE_PATH = "/content/drive/MyDrive/MiniMed_Prime"  #@param {type:"string"}
PROJECT_GIT_URL = ""  #@param {type:"string"}
PROJECT_GIT_REF = "main"  #@param {type:"string"}
PROJECT_ZIP_PATH = ""  #@param {type:"string"}

ASSET_ROOT = "/content/drive/MyDrive/kaggle_assets"  #@param {type:"string"}
OUTPUT_ROOT = "/content/drive/MyDrive/minimed_colab_outputs"  #@param {type:"string"}
CACHE_ROOT = "/content/drive/MyDrive/minimed_colab_cache"  #@param {type:"string"}

DOWNLOAD_MISSING_ASSETS = False  #@param {type:"boolean"}
DOWNLOAD_PRIMEKG = True  #@param {type:"boolean"}
DOWNLOAD_SAPBERT = True  #@param {type:"boolean"}
DOWNLOAD_MEDCPT = True  #@param {type:"boolean"}

MEDREASON_SOURCE = "UCSC-VLAA/MedReason"  #@param {type:"string"}
TRAIN_LIMIT = 200  #@param {type:"integer"}
VALIDATION_LIMIT = 50  #@param {type:"integer"}
TRAIN_MAX_SAVED = 200  #@param {type:"integer"}
VALIDATION_MAX_SAVED = 50  #@param {type:"integer"}
EDGE_MAPPER = "auto"  #@param ["auto", "direct", "heuristic", "llm"]
LLM_MODEL_NAME = "gpt-4o-mini"  #@param {type:"string"}
LLM_DEVICE = "cpu"  #@param ["cpu", "cuda"]
RELATION_FILTER = ""  #@param {type:"string"}
ALLOW_EMPTY_GOLD = False  #@param {type:"boolean"}
BUILD_TRM_ARRAYS = False  #@param {type:"boolean"}
PACKAGE_OUTPUT_ZIP = True  #@param {type:"boolean"}

PROJECT_NAME = "MiniMed_Prime"
WORK_ROOT = "/content/workspaces"
RESET_PROJECT_DIR = True
RUN_NAME = ""


In [ ]:
#@title 2. Mount Drive, materialize the project, and print runtime info
from __future__ import annotations

import os
import platform
import shutil
import subprocess
import sys
import textwrap
import zipfile
from datetime import datetime
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

PROJECT_DIR = Path(WORK_ROOT) / PROJECT_NAME
ASSET_ROOT_PATH = Path(ASSET_ROOT)
OUTPUT_ROOT_PATH = Path(OUTPUT_ROOT)
CACHE_ROOT_PATH = Path(CACHE_ROOT)
TMP_ROOT = Path('/content/tmp')
TMP_ROOT.mkdir(parents=True, exist_ok=True)

for path in [PROJECT_DIR.parent, ASSET_ROOT_PATH, OUTPUT_ROOT_PATH, CACHE_ROOT_PATH]:
    path.mkdir(parents=True, exist_ok=True)

if RESET_PROJECT_DIR and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

def materialize_project_source() -> Path:
    if PROJECT_SOURCE_MODE == 'drive_folder':
        source_dir = Path(PROJECT_DRIVE_PATH)
        if not source_dir.exists():
            raise FileNotFoundError(f'Drive project folder not found: {source_dir}')
        shutil.copytree(source_dir, PROJECT_DIR, dirs_exist_ok=True)
        return PROJECT_DIR

    if PROJECT_SOURCE_MODE == 'git':
        if not PROJECT_GIT_URL.strip():
            raise ValueError('PROJECT_GIT_URL must be set when PROJECT_SOURCE_MODE=git.')
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', PROJECT_GIT_REF, PROJECT_GIT_URL, str(PROJECT_DIR)])
        return PROJECT_DIR

    if PROJECT_SOURCE_MODE == 'zip_bundle':
        zip_path = Path(PROJECT_ZIP_PATH)
        if not zip_path.exists():
            raise FileNotFoundError(f'Project zip not found: {zip_path}')
        PROJECT_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as archive:
            archive.extractall(PROJECT_DIR)
        return PROJECT_DIR

    raise ValueError(f'Unsupported PROJECT_SOURCE_MODE: {PROJECT_SOURCE_MODE}')

project_root = materialize_project_source()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

run_stamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
RUN_NAME = RUN_NAME or f'colab_seed_{run_stamp}'
BUILD_ROOT = OUTPUT_ROOT_PATH / RUN_NAME
RAW_ROOT = BUILD_ROOT / 'raw'
QUALITY_ROOT = BUILD_ROOT / 'quality'
TRM_ROOT = BUILD_ROOT / 'trm_medical'
for path in [BUILD_ROOT, RAW_ROOT, QUALITY_ROOT, TRM_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print('Project root:', project_root)
print('Run name:', RUN_NAME)
print('Build root:', BUILD_ROOT)
print('Python:', sys.version)
print('Platform:', platform.platform())

try:
    import torch
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('Torch probe failed:', exc)

print('Directory snapshot:')
for candidate in [project_root / 'scripts', project_root / 'src', project_root / 'notebooks']:
    print(' -', candidate, 'exists=', candidate.exists())


In [ ]:
#@title 3. Install the minimal dependency set for seed building
import subprocess
import sys

MINIMAL_REQUIREMENTS = [
    'pydantic>=2.7,<3',
    'loguru>=0.7.2',
    'networkx>=3.2',
    'pandas>=2.2',
    'rank-bm25>=0.2.2',
    'requests>=2.32',
    'transformers>=4.44.0',
    'sentence-transformers>=3.0.1',
    'datasets>=2.20.0',
    'huggingface-hub>=0.24.0',
    'accelerate>=0.25.0',
    'einops>=0.8.1',
    'spacy>=3.7,<3.8',
    'scispacy>=0.5.5,<0.7',
]
SCISPACY_MODEL_URL = 'https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_lg-0.5.1.tar.gz'

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-qU', 'pip', 'setuptools', 'wheel'])
for requirement in MINIMAL_REQUIREMENTS:
    print('Installing', requirement)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', requirement])
print('Installing', SCISPACY_MODEL_URL)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', SCISPACY_MODEL_URL])

print('Dependency install complete.')


In [ ]:
#@title 4. Configure caches and load Colab Secrets
import os
from pathlib import Path

os.environ['MINIMED_KAGGLE_ASSETS'] = str(ASSET_ROOT_PATH)
os.environ['HF_HOME'] = str(CACHE_ROOT_PATH / 'hf')
os.environ['TRANSFORMERS_CACHE'] = str(CACHE_ROOT_PATH / 'hf' / 'transformers')
os.environ['PIP_CACHE_DIR'] = str(CACHE_ROOT_PATH / 'pip')
os.environ['TMPDIR'] = str(TMP_ROOT)
os.environ['TEMP'] = str(TMP_ROOT)
os.environ['TMP'] = str(TMP_ROOT)
os.environ['MEDREASON_EDGE_LLM'] = LLM_MODEL_NAME

for env_path in [
    Path(os.environ['HF_HOME']),
    Path(os.environ['TRANSFORMERS_CACHE']),
    Path(os.environ['PIP_CACHE_DIR']),
    Path(os.environ['TMPDIR']),
]:
    env_path.mkdir(parents=True, exist_ok=True)

def load_secret(name: str) -> str | None:
    value = os.environ.get(name)
    if value:
        print(f'{name}: already set in environment')
        return value
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
        print(f'{name}: loaded from Colab Secrets')
        return value
    print(f'{name}: not set')
    return None

for secret_name in ['OPENAI_API_KEY', 'GEMINI_API_KEY', 'NCBI_API_KEY', 'HF_TOKEN']:
    load_secret(secret_name)

print('MINIMED_KAGGLE_ASSETS =', os.environ['MINIMED_KAGGLE_ASSETS'])
print('HF_HOME =', os.environ['HF_HOME'])
print('TRANSFORMERS_CACHE =', os.environ['TRANSFORMERS_CACHE'])


In [ ]:
#@title 5. Resolve assets and optionally download missing PrimeKG / SapBERT / MedCPT
from __future__ import annotations

import json
import requests

# Pre-create the asset layout so KaggleEnv resolves to Drive paths even before downloads finish.
(ASSET_ROOT_PATH / 'primekg').mkdir(parents=True, exist_ok=True)
(ASSET_ROOT_PATH / 'checkpoints' / 'sapbert').mkdir(parents=True, exist_ok=True)
(ASSET_ROOT_PATH / 'checkpoints' / 'medcpt-query').mkdir(parents=True, exist_ok=True)
(ASSET_ROOT_PATH / 'checkpoints' / 'medcpt-article').mkdir(parents=True, exist_ok=True)
(ASSET_ROOT_PATH / 'checkpoints' / 'medcpt-cross').mkdir(parents=True, exist_ok=True)

from scripts import setup_environment as setup_environment
from src.utils.kaggle_env import KaggleEnv

warnings = []
if DOWNLOAD_MISSING_ASSETS:
    session = requests.Session()
    session.headers.update({'User-Agent': 'MiniMedPrimeColab/1.0'})
    if DOWNLOAD_PRIMEKG:
        print('Downloading PrimeKG metadata/files if missing ...')
        setup_environment.download_primekg(
            session=session,
            target_dir=KaggleEnv.path('data/kg/primekg'),
            dry_run=False,
            warnings=warnings,
        )
    if DOWNLOAD_SAPBERT:
        print('Downloading SapBERT if missing ...')
        setup_environment.download_hf_snapshot(
            repo_id=setup_environment.SAPBERT_REPO_ID,
            target_dir=KaggleEnv.path('data/checkpoints/sapbert'),
            dry_run=False,
            warnings=warnings,
        )
    if DOWNLOAD_MEDCPT:
        print('Downloading MedCPT encoders if missing ...')
        setup_environment.download_medcpt(dry_run=False, warnings=warnings)

resolved_paths = {
    'primekg': KaggleEnv.path('data/kg/primekg'),
    'sapbert': KaggleEnv.path('data/checkpoints/sapbert'),
    'medcpt-query': KaggleEnv.path('data/checkpoints/medcpt-query'),
    'medcpt-article': KaggleEnv.path('data/checkpoints/medcpt-article'),
    'medcpt-cross': KaggleEnv.path('data/checkpoints/medcpt-cross'),
}

for name, path in resolved_paths.items():
    print(name, '->', path, 'exists=', path.exists())

if warnings:
    print('Warnings:')
    print(json.dumps(warnings, indent=2))


In [ ]:
#@title 6. Validate scispaCy and resolved runtime paths
import os
from pathlib import Path

import scispacy  # noqa: F401
import spacy

from src.utils.kaggle_env import KaggleEnv

print('spaCy version:', spacy.__version__)
print('scispaCy version:', scispacy.__version__)

nlp = spacy.load('en_core_sci_lg')
print('Loaded en_core_sci_lg with pipes:', nlp.pipe_names[:5])

def contains_any_signature(root: Path, signatures: list[tuple[str, ...]]) -> bool:
    if not root.exists():
        return False
    return any(all((root / part).exists() for part in signature) for signature in signatures)

asset_checks = {
    'data/kg/primekg': [('edges.csv', 'nodes.csv'), ('kg.csv',)],
    'data/checkpoints/sapbert': [('config.json', 'vocab.txt'), ('config.json', 'tokenizer_config.json')],
    'data/checkpoints/medcpt-query': [('config.json', 'tokenizer_config.json')],
    'data/checkpoints/medcpt-article': [('config.json', 'tokenizer_config.json')],
    'data/checkpoints/medcpt-cross': [('config.json', 'tokenizer_config.json')],
}

for logical, signatures in asset_checks.items():
    resolved = KaggleEnv.path(logical)
    signature_ok = contains_any_signature(resolved, signatures)
    print(logical, '->', resolved, 'exists=', resolved.exists(), 'signature_ok=', signature_ok)
    if not signature_ok:
        raise RuntimeError(f'Missing required files for {logical} at {resolved}. Either copy the asset into Drive or enable DOWNLOAD_MISSING_ASSETS.')

if EDGE_MAPPER in {'auto', 'llm'}:
    normalized_model = LLM_MODEL_NAME.strip().lower()
    if normalized_model.startswith('gpt-') and not os.getenv('OPENAI_API_KEY'):
        raise RuntimeError('LLM_MODEL_NAME points to OpenAI, but OPENAI_API_KEY is missing.')
    if 'gemini' in normalized_model and not os.getenv('GEMINI_API_KEY'):
        raise RuntimeError('LLM_MODEL_NAME points to Gemini, but GEMINI_API_KEY is missing.')

print('Runtime validation complete.')


In [ ]:
#@title 7. Build train and validation seed JSONL
from __future__ import annotations

import os
import shlex
import subprocess
import sys

train_seed_path = RAW_ROOT / 'trm_seed_train.jsonl'
validation_seed_path = RAW_ROOT / 'trm_seed_validation.jsonl'

def run_command(command: list[str]) -> None:
    print('$', ' '.join(shlex.quote(part) for part in command))
    completed = subprocess.run(command, cwd=str(PROJECT_DIR), env=os.environ.copy(), text=True)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {command}')

def build_seed(split: str, output_path, limit: int, max_saved: int) -> None:
    command = [
        sys.executable,
        'scripts/prepare_medreason_seed.py',
        '--source', MEDREASON_SOURCE,
        '--split', split,
        '--output-jsonl', str(output_path),
        '--edge-mapper', EDGE_MAPPER,
        '--llm-model-name', LLM_MODEL_NAME,
        '--llm-device', LLM_DEVICE,
    ]
    if limit > 0:
        command.extend(['--limit', str(limit)])
    if max_saved > 0:
        command.extend(['--max-saved', str(max_saved)])
    if RELATION_FILTER.strip():
        command.extend(['--relation-filter', RELATION_FILTER.strip()])
    if ALLOW_EMPTY_GOLD:
        command.append('--allow-empty-gold')
    run_command(command)

build_seed('train', train_seed_path, TRAIN_LIMIT, TRAIN_MAX_SAVED)
build_seed('validation', validation_seed_path, VALIDATION_LIMIT, VALIDATION_MAX_SAVED)

print('Train seed:', train_seed_path, 'exists=', train_seed_path.exists())
print('Validation seed:', validation_seed_path, 'exists=', validation_seed_path.exists())


In [ ]:
#@title 8. Split train and validation into goldish / silver / reject tiers
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys

train_quality_dir = QUALITY_ROOT / 'train'
validation_quality_dir = QUALITY_ROOT / 'validation'

def run_quality_filter(input_jsonl, output_dir) -> dict:
    command = [
        sys.executable,
        'scripts/filter_seed_quality.py',
        '--input-jsonl', str(input_jsonl),
        '--output-dir', str(output_dir),
    ]
    print('$', ' '.join(shlex.quote(part) for part in command))
    completed = subprocess.run(
        command,
        cwd=str(PROJECT_DIR),
        env=os.environ.copy(),
        text=True,
        capture_output=True,
    )
    print(completed.stdout)
    if completed.returncode not in (0, 1):
        raise RuntimeError(completed.stderr or completed.stdout)
    summary_path = output_dir / 'seed_quality_summary.json'
    return json.loads(summary_path.read_text(encoding='utf-8'))

train_summary = run_quality_filter(train_seed_path, train_quality_dir)
validation_summary = run_quality_filter(validation_seed_path, validation_quality_dir)

print('Train summary:')
print(json.dumps(train_summary, indent=2, sort_keys=True))
print('Validation summary:')
print(json.dumps(validation_summary, indent=2, sort_keys=True))


In [ ]:
#@title 9. Inspect the filtered outputs
from __future__ import annotations

import json
from pathlib import Path

def preview_jsonl(path: Path, max_rows: int = 2) -> None:
    print('\n===', path, '===')
    if not path.exists():
        print('missing')
        return
    rows = [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    print('rows:', len(rows))
    for row in rows[:max_rows]:
        quality = row.get('quality', {})
        metadata = row.get('metadata', {})
        edge_mapping = metadata.get('edge_mapping', {})
        print({
            'group_id': row.get('group_id'),
            'gold_edge_count': len(row.get('gold_edge_ids', [])),
            'quality_tier': quality.get('tier'),
            'quality_reasons': quality.get('reasons'),
            'edge_mapper_backend': edge_mapping.get('backend_used'),
            'entity_linker_backend': row.get('evidence', {}).get('metadata', {}).get('entity_linker_backend'),
            'primekg_backend': row.get('evidence', {}).get('metadata', {}).get('backend_used', {}).get('primekg'),
        })

preview_jsonl(train_quality_dir / 'trm_seed_goldish.jsonl')
preview_jsonl(train_quality_dir / 'trm_seed_silver.jsonl')
preview_jsonl(train_quality_dir / 'trm_seed_rejects.jsonl')
preview_jsonl(validation_quality_dir / 'trm_seed_goldish.jsonl')


In [ ]:
#@title 10. Optional: build TRM arrays from the goldish split
from __future__ import annotations

import os
import shlex
import subprocess
import sys

if BUILD_TRM_ARRAYS:
    def build_trm(split_name: str, input_jsonl) -> None:
        command = [
            sys.executable,
            'scripts/build_trm_dataset.py',
            '--input-jsonl', str(input_jsonl),
            '--output-dir', str(TRM_ROOT),
            '--split', split_name,
        ]
        print('$', ' '.join(shlex.quote(part) for part in command))
        completed = subprocess.run(command, cwd=str(PROJECT_DIR), env=os.environ.copy(), text=True)
        if completed.returncode != 0:
            raise RuntimeError(f'build_trm_dataset failed for {split_name}')

    build_trm('train', train_quality_dir / 'trm_seed_goldish.jsonl')
    build_trm('val', validation_quality_dir / 'trm_seed_goldish.jsonl')
    print('TRM arrays saved under', TRM_ROOT)
else:
    print('BUILD_TRM_ARRAYS=False, skipping TRM dataset materialization.')


In [ ]:
#@title 11. Package the output folder for download or Kaggle upload
from __future__ import annotations

import shutil
from pathlib import Path

if PACKAGE_OUTPUT_ZIP:
    archive_base = BUILD_ROOT.parent / BUILD_ROOT.name
    archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=str(BUILD_ROOT))
    print('Packaged archive:', archive_path)
else:
    print('PACKAGE_OUTPUT_ZIP=False, skipping zip packaging.')

print('Final output tree root:', BUILD_ROOT)
for child in sorted(BUILD_ROOT.iterdir()):
    print(' -', child)


## Next Step

If the `goldish` counts look healthy, upload these files to Kaggle and train there:
- `quality/train/trm_seed_goldish.jsonl`
- `quality/validation/trm_seed_goldish.jsonl`

If `goldish` is too small, inspect `silver` and `reject` first. The common causes are:
- `entity_linker_backend=rule_based` because scispaCy did not load correctly
- `edge_mapper_backend=heuristic` because the LLM backend was not actually active
- `primekg_backend=empty_graph` because PrimeKG did not resolve correctly
